# Lesson 15: Language Modeling & Custom CBoW / Skip-Gram Embeddings

In this notebook, you will:
1. Train a **CBoW** word embedding model from scratch in PyTorch.
2. Complete the **Lab**: Modify the architecture to train a **Skip-Gram** model and inspect nearest-neighbor words.

In [ ]:
import torch
import torch.nn as nn

corpus = (
    "we are about to study the idea of a computational process "
    "computational processes are abstract beings that inhabit computers "
    "as they evolve processes manipulate other abstract things called data"
).split()

vocab = sorted(set(corpus))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}

# Build CBoW and Skip-Gram pairs (window=2)
cbow_data, skip_data = [], []
for i in range(2, len(corpus) - 2):
    ctx = [w2i[corpus[i-2]], w2i[corpus[i-1]], w2i[corpus[i+1]], w2i[corpus[i+2]]]
    tgt = w2i[corpus[i]]
    cbow_data.append((ctx, tgt))
    for c in ctx:
        skip_data.append((tgt, c))

class CBoW(nn.Module):
    def __init__(self, vocab_size, emb_dim=16):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.fc = nn.Linear(emb_dim, vocab_size)
    def forward(self, ctx):
        return self.fc(self.emb(ctx).mean(dim=1))

model = CBoW(len(vocab))
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()
X_cbow = torch.tensor([x for x, _ in cbow_data])
y_cbow = torch.tensor([y for _, y in cbow_data])

for ep in range(80):
    opt.zero_grad()
    loss = loss_fn(model(X_cbow), y_cbow)
    loss.backward()
    opt.step()
print(f"Final CBoW Loss: {loss.item():.4f}")

# Lab: Skip-Gram Model
class SkipGram(nn.Module):
    def __init__(self, vocab_size, emb_dim=16):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.fc = nn.Linear(emb_dim, vocab_size)
    def forward(self, center):
        return self.fc(self.emb(center))

sg = SkipGram(len(vocab))
opt_sg = torch.optim.Adam(sg.parameters(), lr=0.05)
X_sg = torch.tensor([x for x, _ in skip_data])
y_sg = torch.tensor([y for _, y in skip_data])
for ep in range(80):
    opt_sg.zero_grad()
    loss_sg = loss_fn(sg(X_sg), y_sg)
    loss_sg.backward()
    opt_sg.step()
print(f"Final Skip-Gram Loss: {loss_sg.item():.4f}")

## Part 2: Nearest-Neighbor Word Inspection

In [ ]:
import torch.nn.functional as F
emb_matrix = sg.emb.weight.detach()
target_vec = emb_matrix[w2i["computational"]].unsqueeze(0)
sims = F.cosine_similarity(target_vec, emb_matrix)
top_ids = sims.argsort(descending=True)[:4].tolist()
print("Nearest neighbors to 'computational':", [i2w[i] for i in top_ids])